# Fitbit Fitness Tracker Data - Project Setup

This notebook records the environment, input inventory, checksums,
parsing policy, loader plan, and access assumptions shared by
Simple and Advanced EDA.

## Step 0: Confirm setup and permitted inputs

### 0.1 Project root and existing setup

Locate the repository root independently of the notebook's working directory.

Check the existing Git repository, virtual environment, CI workflow,
and registered Fitbit kernel. Reuse this setup without reinstalling
packages or resetting project files.

Record the current branch, baseline commit, and uncommitted changes.
The baseline commit may not include the notebook changes currently being made.

In [1]:
import sys
import platform
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Find the repository root, even when the notebook runs inside EDA/. 
cwd = Path.cwd().resolve() 


PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / ".git").exists()
        and (folder / "requirements.txt").is_file()
    ),
    None,
)

if PROJECT_ROOT is None: 
    raise FileNotFoundError("Could not find the project root containing .git and requirements.txt")


def run_git(*arguments):
    """Run a read-only Git command in this project."""
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), *arguments],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

# These commands inspect Git; they do not change the repository.
git_branch = run_git("branch", "--show-current")
git_commit = run_git("rev-parse", "HEAD")
git_changes = run_git("status", "--short")

setup_paths = {
    "Git metadata": PROJECT_ROOT / ".git",
    "Virtual environment": PROJECT_ROOT / ".venv",
    "CI workflow": PROJECT_ROOT / ".github" / "workflows" / "ci.yml",
    "Requirements": PROJECT_ROOT / "requirements.txt",
    "Dataset folder": PROJECT_ROOT / "Dataset",
    "EDA folder": PROJECT_ROOT / "EDA",
    "Fitbit kernel specification": (
        PROJECT_ROOT
        / ".venv"
        / "share"
        / "jupyter"
        / "kernels"
        / "fitbit-fitness-tracker"
        / "kernel.json"
    ),
}
    
print("Working directory:", cwd)
print("Project root     :", PROJECT_ROOT)
print("Branch           :", git_branch)
print("Baseline commit  :", git_commit)

print("\nExisting setup:")
for label, path in setup_paths.items():
    print(f"{label:<30} exists: {path.exists()}")

print("\nUncommitted changes:")
print(git_changes or "Working tree clean")

Working directory: D:\Projects\AI and ML\FitBit Fitness Tracker Data\EDA
Project root     : D:\Projects\AI and ML\FitBit Fitness Tracker Data
Branch           : eda/1-simple-eda
Baseline commit  : 674a44b24ca71ae2d20e97424da3f4cba5e19dbf

Existing setup:
Git metadata                   exists: True
Virtual environment            exists: True
CI workflow                    exists: True
Requirements                   exists: True
Dataset folder                 exists: True
EDA folder                     exists: True
Fitbit kernel specification    exists: True

Uncommitted changes:
AM EDA/Project_Setup.ipynb
M  "EDA/Simple EDA.ipynb"
A  EDA/input_manifest.csv


### 0.2 Environment verification and CI scope

Confirm that the notebook's Python process belongs to this project's .venv. 

Record Python, OS, and installed direct-dependency versions.
Compare package versions with requirements.txt.

CI checks: 
- Dependency installation.
- Dependency compatibility using pip check.
- Import of the selected analysis and notebook packages.

CI does not load the Fitbit dataset, execute this notebook,
or establish that EDA findings are correct.

In [2]:
# Verify the actual running Python environment.
is_virtual_environment = sys.prefix != sys.base_prefix
is_project_environment = (
    Path(sys.prefix).resolve()
    == (PROJECT_ROOT / ".venv").resolve()
)

print("Python version     :", sys.version.split()[0])
print("Python executable  :", sys.executable)
print("OS                 :", platform.platform())
print("Virtual environment:", is_virtual_environment)
print("Project .venv      :", is_project_environment)

if not is_project_environment:
    raise RuntimeError("Select the project's Fitbit kernel before continuing.")

# Read the direct-version pins already recorded in requirements.txt  
requirements_path = PROJECT_ROOT / "requirements.txt"
required_versions = {}

for line in requirements_path.read_text(encoding="utf-8-sig").splitlines():
    line = line.strip()

    if not line or line.startswith("#"):
        continue

    if "==" in line:
        package, pinned_version = line.split("==", 1)
        required_versions[package.strip()] = pinned_version.strip()

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "jupyterlab",
    "ipykernel",
]

package_records = []

print("\nInstalled packages versus requirements:")
for package in packages:
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = "NOT INSTALLED"

    pinned_version = required_versions.get(package, "NOT PINNED")
    matches_pin = installed_version == pinned_version

    package_records.append(
        {
            "package": package,
            "installed": installed_version,
            "required": pinned_version,
            "matches_pin": matches_pin,
        }
    )

    print(
        f"{package:<12} installed={installed_version:<15} "
        f"required={pinned_version:<15} match={matches_pin}"
    )


Python version     : 3.12.10
Python executable  : D:\Projects\AI and ML\FitBit Fitness Tracker Data\.venv\Scripts\python.exe
OS                 : Windows-11-10.0.26200-SP0
Virtual environment: True
Project .venv      : True

Installed packages versus requirements:
pandas       installed=3.0.6           required=3.0.6           match=True
numpy        installed=2.5.4           required=2.5.4           match=True
matplotlib   installed=3.11.2          required=3.11.2          match=True
seaborn      installed=0.13.2          required=0.13.2          match=True
jupyterlab   installed=4.6.4           required=4.6.4           match=True
ipykernel    installed=7.4.0           required=7.4.0           match=True


### 0.3 Identify the local inputs

The local dataset contains two export folders, labelled P1 and P2.
Folder labels identify exports; actual observation dates remain to be verified.

Record CSV filenames, export membership, and byte sizes.
Save this metadata in EDA/input_manifest.csv.

Reference pages:
- Kaggle: https://www.kaggle.com/datasets/arashnic/fitbit
- Original study deposit: https://zenodo.org/records/53894


In [3]:
import pandas as pd 

DATA_ROOT = PROJECT_ROOT / "Dataset"

PERIOD_DIRS = {
    "P1": (
        DATA_ROOT
        / "mturkfitbit_export_3.12.16-4.11.16"
        / "Fitabase Data 3.12.16-4.11.16"
    ),
    "P2": (
        DATA_ROOT
        / "mturkfitbit_export_4.12.16-5.12.16"
        / "Fitabase Data 4.12.16-5.12.16"
    ),
}

for source_folder, folder in PERIOD_DIRS.items():
    if not folder.is_dir():
        raise FileNotFoundError(
            f"{source_folder} export folder not found: {folder}"
        )

def identify_source_folder(path):
    """Return the configured export label for a discovered file."""
    for source_folder, folder in PERIOD_DIRS.items():
        if path.parent.resolve() == folder.resolve():
            return source_folder

    raise ValueError(f"CSV outside the configured export folders: {path}")


csv_paths = sorted(DATA_ROOT.rglob("*.csv"))

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under: {DATA_ROOT}")

# source_folder means the export label P1/P2.
# source_file is the full path relative to Dataset/, using portable slashes.
input_inventory = pd.DataFrame(
    [
        {
            "source_folder": identify_source_folder(path),
            "source_file": path.relative_to(DATA_ROOT).as_posix(),
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
        }
        for path in csv_paths
    ]
)

input_inventory["size_mb"] = (
    input_inventory["size_bytes"] / 1_000_000
).round(2)

period_summary = (
    input_inventory.groupby("source_folder")
    .agg(
        csv_files=("source_file", "size"),
        size_bytes=("size_bytes", "sum"),
    )
)

period_summary["size_mb"] = (
    period_summary["size_bytes"] / 1_000_000
).round(2)

print("CSV files found:", len(input_inventory))
print("Total bytes    :", input_inventory["size_bytes"].sum())

display(period_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(input_inventory)

CSV files found: 29
Total bytes    : 616796545


,csv_files,size_bytes,size_mb
source_folder,,,
P1,11,278798829,278.8
P2,18,337997716,338.0


,source_folder,source_file,file_name,size_bytes,size_mb
0,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyActivity_merged.csv,dailyActivity_merged.csv,51260,0.05
1,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/heartrate_seconds_merged.csv,heartrate_seconds_merged.csv,41069585,41.07
2,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/hourlyCalories_merged.csv,hourlyCalories_merged.csv,872698,0.87
3,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/hourlyIntensities_merged.csv,hourlyIntensities_merged.csv,971718,0.97
4,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/hourlySteps_merged.csv,hourlySteps_merged.csv,865300,0.87
5,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteCaloriesNarrow_merged.csv,minuteCaloriesNarrow_merged.csv,72461949,72.46
6,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteIntensitiesNarrow_merged.csv,minuteIntensitiesNarrow_merged.csv,50523569,50.52
7,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteMETsNarrow_merged.csv,minuteMETsNarrow_merged.csv,51970880,51.97
8,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteSleep_merged.csv,minuteSleep_merged.csv,9316325,9.32
9,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteStepsNarrow_merged.csv,minuteStepsNarrow_merged.csv,50692143,50.69


#### Input-file checksums

Record a SHA-256 checksum for every input CSV.

The checksums identify the file contents used in this analysis.
They allow future runs to detect content changes, but do not prevent
editing or establish data accuracy.

This operation reads file bytes without changing the CSV files.

In [4]:
import hashlib 

def calculate_sha256(path): 
    # Calculate a file checksum without modifying its contents. 
    with path.open("rb") as file: 
        return hashlib.file_digest(file, "sha256").hexdigest() 

input_inventory["sha256"] = [
      calculate_sha256(DATA_ROOT / source_file) 
      for source_file in input_inventory["source_file"]
]

print("Files fingerprinted:", len(input_inventory)) 

# Preview metadata only—not participant records.
display(
    input_inventory[
        ["source_folder", "source_file", "size_bytes", "sha256"]
    ].head(3)
)

Files fingerprinted: 29


,source_folder,source_file,size_bytes,sha256
0,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Da...,51260,b56bf67d7eb322da48d74680aa7ea6552aa4cdc83759ca...
1,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Da...,41069585,feabb4f4dcfb4a64f1616a01a940fef4cd7c516d493d46...
2,P1,mturkfitbit_export_3.12.16-4.11.16/Fitabase Da...,872698,3fa7b2bd6fce95016253c3cbfc71eed32e7291da5bd172...


**Save the metadata manifest:** 

In [5]:
manifest_path = PROJECT_ROOT / "EDA" / "input_manifest.csv"

# Store exact byte sizes; rounded MB values are only for display.
manifest_table = (
    input_inventory[["source_folder", "source_file", "size_bytes", "sha256"]]
    .sort_values(["source_folder", "source_file"])
    .reset_index(drop=True)
)

if manifest_path.exists():
    # Preserve an existing manifest rather than silently replacing it.
    existing_manifest = pd.read_csv(
        manifest_path,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    if list(existing_manifest.columns) != list(manifest_table.columns):
        raise ValueError(
            "Existing manifest uses a different format. Review it before replacing it."
        )

    existing_manifest = (
        existing_manifest
        .sort_values(["source_folder", "source_file"])
        .reset_index(drop=True)
    )

    if not existing_manifest.equals(manifest_table.astype("string")):
        raise ValueError(
            "Current filenames, export membership, or sizes differ from the manifest."
        )

    print("Existing manifest matches the current filename/size inventory.")
else:
    manifest_table.to_csv(
        manifest_path,
        index=False,
        encoding="utf-8",
        mode="x",
    )
    print("Metadata manifest created:", manifest_path)

Existing manifest matches the current filename/size inventory.


### 0.4 Parsing policy

Initial ingestion will preserve source field values as strings.

- IDs remain text identifiers.
- Empty strings and tokens such as NA, N/A, and NULL remain unchanged until their meanings are examined.
- Zero remains distinct from missing data.
- Date/time strings remain unchanged during initial ingestion.
- Start with UTF-8/BOM-compatible decoding and strict decoding errors. If decoding fails, investigate rather than silently replacing characters.
- Use comma separation provisionally and validate the file structure.
- Do not skip blank records or parser-reported malformed rows.
- Check duplicate/empty headers and record widths during loader validation.

Candidate date formats to verify:
- Date: month/day/year.
- Datetime: month/day/year followed by 12-hour time and AM/PM.

Later derived columns may use:
- Nullable integers for verified integral counts.
- Nullable floating-point values for numeric measurements.
- Explicit datetime formats after inspection.

Retain original strings alongside relevant conversions.
Do not round analytical values or assign an unknown timezone.

Parser settings are a planned policy, not evidence that every file
has already passed parsing or structural validation.

In [6]:
READ_POLICY = {
    "sep": ",",
    "encoding": "utf-8-sig",
    "encoding_errors": "strict",
    "dtype": "string",
    "na_filter": False,
    "parse_dates": False,
    "skip_blank_lines": False,
    "on_bad_lines": "error",
}


# Proposed formats only. Do not apply them until inspected and verified.
CANDIDATE_DATE_FORMATS = {
    "date": "%m/%d/%Y",
    "datetime": "%m/%d/%Y %I:%M:%S %p",
}

print("Initial CSV parsing policy:")
for setting, value in READ_POLICY.items():
    print(f"{setting:<20}: {value}")

print("\nCandidate date formats:")
for field_type, date_format in CANDIDATE_DATE_FORMATS.items():
    print(f"{field_type:<20}: {date_format}")

Initial CSV parsing policy:
sep                 : ,
encoding            : utf-8-sig
encoding_errors     : strict
dtype               : string
na_filter           : False
parse_dates         : False
skip_blank_lines    : False
on_bad_lines        : error

Candidate date formats:
date                : %m/%d/%Y
datetime            : %m/%d/%Y %I:%M:%S %p


### 0.5 Loader and row-provenance plan

Use the inventoried files as the input registry.

The loader will:
- Select a configured export and filename explicitly.
- Begin with dailyActivity_merged.csv.
- Read one file at a time.
- Offer chunked reading for large files.
- Preserve the initial string representation and parsing policy.
- Validate headers, record widths, and provenance-column collisions.
- Retain source_folder, source_file, and source_row.

Provenance definitions:
- source_folder: export label, P1 or P2.
- source_file: path relative to Dataset/.
- source_row: one-based data-record number within the source file,
  excluding the header; not necessarily a physical text-line number.

Chunked loading must continue source_row numbering across chunks.

The loader is planned here; it has not been executed or validated.
No participant records are loaded in Step 0.

In [7]:
# Build a registry from the files we actually discovered.
INPUT_FILES = {}

for path in csv_paths:
    source_folder = identify_source_folder(path)
    key = (source_folder, path.name)

    if key in INPUT_FILES:
        raise ValueError(f"Ambiguous input registry key: {key}")

    INPUT_FILES[key] = path

LOADER_PLAN = {
    "first_file": "dailyActivity_merged.csv",
    "initial_exports": ["P1", "P2"],
    "default_chunk_rows": 100_000,
    "provenance_columns": [
        "source_folder",
        "source_file",
        "source_row",
    ],
    "implementation_status": "Planned; implement and validate in Step 2",
}

# Verify proposed first inputs exist without reading their records.
for source_folder in LOADER_PLAN["initial_exports"]:
    key = (source_folder, LOADER_PLAN["first_file"])

    if key not in INPUT_FILES:
        raise FileNotFoundError(f"Proposed first input not found: {key}")

    print(
        source_folder,
        INPUT_FILES[key].relative_to(DATA_ROOT).as_posix(),
    )

print("\nRegistered CSV files:", len(INPUT_FILES))
print("Loader status:", LOADER_PLAN["implementation_status"])

P1 mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyActivity_merged.csv
P2 mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/dailyActivity_merged.csv

Registered CSV files: 29
Loader status: Planned; implement and validate in Step 2


### 0.6 Data handling

- Keep raw CSVs unchanged in `Dataset/`.
- Keep `Dataset/` and `.venv/` excluded from Git.
- Before committing notebooks, review outputs and hide original participant IDs,
  log IDs, and unnecessary local paths.
- Record previews may remain in shared notebooks when person IDs use preview
  codes and log IDs are hidden.
- Share aggregate results and the input manifest.

**Updated 2026-10-17:** Keep redacted record previews. This replaces the earlier
rule that removed all individual health records from shared outputs.

### Step 0 - Final review

Before committing:

- [ ] Restart the kernel and run all setup cells successfully.
- [ ] Confirm the project environment and package versions pass their checks.
- [ ] Confirm the manifest records all 29 CSVs, their sizes, and SHA-256 checksums.
- [ ] Confirm `Dataset/` and `.venv/` are ignored and contain no tracked files.
- [ ] Review notebook outputs before sharing.

Next: define the analysis scope in `Simple EDA.ipynb`.